# Sentinel-2 monthly NDVI time-series (Earth Search)

Shows the **raster `aggregate=`** path: pull Sentinel-2 red + NIR over a farm for a season, reduce per month into per-window COGs, then compute NDVI per month. Offline cells below build the aggregation request; the live pull is a recipe.

In [ ]:
from earthlens.aggregate import AggregationConfig

# Monthly mean windows; one COG per (collection, month) is written.
agg = AggregationConfig(freq="1MS", op="mean", out_dir="out/farm/monthly")
print(agg.freq, agg.op)

## Pull + aggregate (live)

```python
from earthlens.earthlens import EarthLens
from earthlens.aggregate import AggregationConfig

el = EarthLens(
    data_source="earth-search",
    start="2024-04-01", end="2024-09-30",
    variables={"sentinel-2-l2a": ["red", "nir"]},
    lat_lim=[40.40, 40.45], lon_lim=[-3.72, -3.67],
    path="out/farm",
)
monthly = el.download(aggregate=AggregationConfig(freq="1MS", op="mean",
                                                  out_dir="out/farm/monthly"))
# `monthly` is one COG per month, named sentinel-2-l2a_mean_1MS_<YYYYMMDD>.tif
```

## NDVI per window

```python
import numpy as np
from pyramids.dataset import Dataset

for cog in monthly:
    ds = Dataset.read_file(str(cog))
    arr = ds.read_array()              # (band, y, x): band 0 = red, 1 = nir
    red, nir = arr[0].astype(float), arr[1].astype(float)
    ndvi = (nir - red) / (nir + red + 1e-9)
    print(cog.name, float(np.nanmean(ndvi)))
```

Plotting the per-month NDVI mean gives the seasonal greening curve.